In [ ]:
# ============================================================
# AI-BASED PRONUNCIATION & FLUENCY ASSESSMENT SYSTEM
# GOOGLE COLAB - ONE CELL
# ============================================================

!pip -q install gradio==5.44.1 faster-whisper

import re
import difflib
import gradio as gr
from faster_whisper import WhisperModel

print("Loading speech model...")

model = WhisperModel(
    "tiny",
    device="cpu",
    compute_type="int8"
)

print("✅ Model loaded!")


def clean_text(text):

    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        "",
        text
    )

    return text.strip()


def assess_pronunciation(
    expected,
    audio
):

    if not expected.strip():

        return "⚠️ Please enter the expected sentence."

    if audio is None:

        return "⚠️ Please record or upload your pronunciation."

    try:

        # Speech recognition
        segments, info = model.transcribe(
            audio,
            beam_size=1
        )

        segments = list(segments)

        spoken = " ".join(
            s.text.strip()
            for s in segments
        ).strip()

        if not spoken:

            return "❌ Could not recognize speech."

        expected_clean = clean_text(
            expected
        )

        spoken_clean = clean_text(
            spoken
        )

        # Word comparison
        expected_words = expected_clean.split()

        spoken_words = spoken_clean.split()

        matcher = difflib.SequenceMatcher(
            None,
            expected_words,
            spoken_words
        )

        accuracy = int(
            matcher.ratio() * 100
        )

        # Fluency
        duration = info.duration

        wpm = int(
            len(spoken_words) /
            max(duration / 60, 0.01)
        )

        if 90 <= wpm <= 170:
            fluency_score = 95
        elif 70 <= wpm <= 190:
            fluency_score = 80
        else:
            fluency_score = 60

        overall = int(
            (
                accuracy +
                fluency_score
            ) / 2
        )

        if overall >= 90:
            level = "Excellent 🌟"
        elif overall >= 75:
            level = "Good 👍"
        elif overall >= 60:
            level = "Average 🙂"
        else:
            level = "Needs Practice 📚"

        # Difference
        matcher = difflib.SequenceMatcher(
            None,
            expected_words,
            spoken_words
        )

        differences = []

        for tag, i1, i2, j1, j2 in matcher.get_opcodes():

            if tag != "equal":

                differences.append(
                    f"Expected: {' '.join(expected_words[i1:i2])} | "
                    f"Heard: {' '.join(spoken_words[j1:j2])}"
                )

        if not differences:

            differences_text = "All words matched successfully! ✅"

        else:

            differences_text = "\n".join(
                f"- {d}"
                for d in differences
            )

        result = f"""
# 🗣️ Pronunciation & Fluency Report

## 🏆 Overall Score

# {overall}/100

### Performance

**{level}**

---

## 📌 Expected Sentence

> {expected}

## 🎙️ Recognized Speech

> {spoken}

---

## 📊 Analysis

| Metric | Score |
|---|---|
| Word Accuracy | {accuracy}% |
| Fluency Score | {fluency_score}% |
| Speaking Speed | {wpm} WPM |
| Overall Score | {overall}% |

---

## 🔎 Word Comparison

{differences_text}

---

## 💡 Improvement Tips

"""

        if accuracy < 75:
            result += "- Practice the sentence slowly and clearly.\\n"
        else:
            result += "- Your word recognition accuracy is good.\\n"

        if wpm < 90:
            result += "- Try to maintain a natural speaking speed.\\n"
        elif wpm > 180:
            result += "- Slow down slightly for better clarity.\\n"
        else:
            result += "- Your speaking speed is within a good range.\\n"

        result += """
- Practice difficult words individually.
- Record yourself and compare repeated attempts.
- Focus on clear pronunciation rather than speaking very fast.
"""

        return result

    except Exception as e:

        return f"❌ Error: {e}"


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Pronunciation Assessment"
) as app:

    gr.Markdown("""
# 🗣️ AI-Based Pronunciation & Fluency Assessment System

### Compare your spoken sentence with the expected sentence.

**Expected Text + Voice → Speech Recognition → Word Comparison → Fluency Score**
""")

    expected = gr.Textbox(
        label="📖 Expected Sentence",
        placeholder="Example: Artificial intelligence is changing the world.",
        lines=3
    )

    audio = gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="🎙️ Speak the Sentence"
    )

    button = gr.Button(
        "🔍 Assess Pronunciation",
        variant="primary"
    )

    output = gr.Markdown()

    button.click(
        assess_pronunciation,
        inputs=[
            expected,
            audio
        ],
        outputs=output
    )

    gr.Markdown("""
---
### TSA Concepts

Speech-to-Text • Text Comparison • Word Accuracy •
Fluency Analysis • Pronunciation Assessment
""")


print("🚀 Launching public URL...")

app.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/75.5M [00:00<?, ?B/s]

✅ Model loaded!
🚀 Launching public URL...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f6198558bf34de5483.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
